# 06 - Smoothed latent vectors

The 7-day baseline clusters the backward 7-day mean of the daily mean and standard deviation of the price. This notebook gives the VAE-LSTM latents the same help. No model is trained: it reads the latents saved by notebook 03, clusters them and saves only the regimes. Every metric is computed afterwards by `scripts/06_regime_metrics.py`, the same way as for the runs of notebooks 03 and 05.

For every trained model in `studies` and every k in `k_values`, k-means runs on one input (the latents as notebook 03 saved them, with the daily mean and std, is the run of notebook 03 itself):
- `latents 7d + level 7d`, daily models: the latents and the daily mean and std, all replaced by their backward 7-day mean (the day and the 6 days before it; the first 6 days average the days available).
- `latents + level 7d`, weekly models: the latents as they are and the backward 7-day mean of the daily mean and std: the weekly scalogram of day t already covers day t and the 6 days before it, so its latents are not averaged again (that would use 13 days), and the price gets the same 7-day mean as in the 7-day baseline. The two differ only in the scalogram.

As in notebook 03, k-means is fitted on training days, assigns every day, and regimes are numbered by ascending mean real price. Each model is read with the scalograms it was trained on (`scalogram_set` in its `config.json`), so the weekly models use the weekly days, split and price columns. The regimes are saved in `experiments/vae_lstm/latent_smoothing/regimes.csv` (one row per date and one column per input; a column is empty on the dates its scalogram set does not have, the first 6 days for the weekly set) and `inputs.csv` (what each column is).

<d.benavidess@uniandes.edu.co>

In [6]:
import sys
sys.path.append("..")

import json
from pathlib import Path

import numpy as np
import pandas as pd

from src.data import load_context, date_split
from src.clustering import level_columns, fit_clusters, order_regimes
from src import experiments as exp

In [7]:
import warnings
warnings.filterwarnings("ignore")

## 0. Configuration

`studies` are the study folders of notebook 03 whose trained models are read (a study that does not exist yet is skipped). Each trained model is read once, from the folder named by `trained_in`, so its `artifacts/latents.npy` must be on this computer. `scalogram_sets` are the scalogram sets whose models are read; a model trained on another set is skipped. Models of `weekly_set` get the weekly inputs (section 2).

In [ ]:
scalogram_sets = ["real_price_log", "real_price_log_weekly"]
studies = ["vae_lstm/hyperparameters", "vae_lstm/lam_beta_grid", "vae_lstm/dimensions", "vae_lstm/weekly"]
weekly_set = "real_price_log_weekly"
k_values = [3, 4, 5]
window = 7
out_dir = exp.EXPERIMENTS_DIR / "vae_lstm" / "latent_smoothing"

## 1. Data and trained models

In [9]:
sets = {}
for scalogram_set in scalogram_sets:
    metadata = pd.read_csv(Path("../data/processed/scalograms") / scalogram_set / "scalogram_metadata.csv",
                           parse_dates=["start_datetime", "end_datetime"])
    context = load_context(Path("../data/processed"), metadata)
    sets[scalogram_set] = {"metadata": metadata, "dates": context["date"].dt.date,
                           "train_idx": date_split(context["date"])[0],
                           "price": context["precio_real_cop_kwh"].to_numpy()}

models = {}
for study in studies:
    for path in sorted((exp.EXPERIMENTS_DIR / study).glob("*/config.json")):
        cfg = json.loads(path.read_text())
        if cfg.get("scalogram_set") in sets and cfg.get("trained_in"):
            models.setdefault(cfg["trained_in"], {"study": cfg["study"], "scalogram_set": cfg["scalogram_set"],
                                                  "z_dim": cfg["z_dim"], "lam": cfg["lam"], "beta": cfg["beta"],
                                                  "seed": cfg["seed"]})
models = pd.DataFrame.from_dict(models, orient="index").rename_axis("run_id").reset_index()

## 2. Clustering

`zscore` scales the price columns with the mean and standard deviation of the training days of the model's scalogram set, as notebook 03 does; the latents are used as they are. `backward_mean` is the backward `window`-day mean of each column.

In [10]:
def zscore(x, train_idx):
    x = np.asarray(x, dtype=np.float64)
    return (x - x[train_idx].mean(axis=0)) / x[train_idx].std(axis=0)


def backward_mean(x):
    return pd.DataFrame(x).rolling(window, min_periods=1).mean().to_numpy()


columns, inputs = {}, []
for scalogram_set, s in sets.items():
    set_models = models[models["scalogram_set"] == scalogram_set] if len(models) else models
    if set_models.empty:
        continue
    train_idx = s["train_idx"]
    level = s["metadata"][list(level_columns(s["metadata"]))].to_numpy(dtype=np.float64)
    level_7d = zscore(backward_mean(level), train_idx)
    features = "latents + level 7d" if scalogram_set == weekly_set else "latents 7d + level 7d"
    for _, model in set_models.iterrows():
        latents = np.load(exp.EXPERIMENTS_DIR / model["run_id"] / "artifacts" / "latents.npy").astype(np.float64)
        # the weekly latents already cover 7 days, so only the daily ones get the backward 7-day mean
        x = np.hstack([latents if scalogram_set == weekly_set else backward_mean(latents), level_7d])
        for k in k_values:
            name = f"c{len(inputs):03d}"
            regimes = order_regimes(fit_clusters(x, train_idx, k, int(model["seed"]))[0], s["price"])
            columns[name] = pd.Series(regimes, index=s["dates"], dtype="Int64")
            inputs.append({"column": name, "model": "vae_lstm", **model.drop("scalogram_set").to_dict(), "k": k,
                           "features": features, "smoothed": True})

out_dir.mkdir(parents=True, exist_ok=True)
pd.DataFrame(columns).sort_index().rename_axis("date").reset_index().to_csv(out_dir / "regimes.csv", index=False)
pd.DataFrame(inputs).to_csv(out_dir / "inputs.csv", index=False)
print(f"{len(models)} models, {len(inputs)} regime columns")

27 models, 216 regime columns
